In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import boto3
import icechunk
import matplotlib.pyplot as plt
import xarray as xr

from srm import catalog

In [3]:
def open_icechunk(path):
    bucket, prefix = path.replace("s3://", "").split("/", 1)
    storage = icechunk.s3_storage(bucket=bucket, prefix=prefix)
    repo = icechunk.Repository.open(storage)
    session = repo.readonly_session("main")

    ds = xr.open_dataset(session.store, engine="zarr", chunks={})
    return ds

In [8]:
def get_fname(var, scenario, version="test015_benchmark", ens="007"):
    fname = (
        "s3://carbonplan-scratch/srm/outputs/qa/"
        + version
        + "/"
        + scenario
        + "/CESM2-WACCM/"
        + var
        + "/"
        + ens
        + "/lat-32.64to-26.58_lon25.05to31.39/*/"
        + scenario
        + ".icechunk/"
    )
    
    return fname

In [9]:
def resolve_s3_glob(path):
    """Resolve a single * wildcard in an S3 path to a real path."""
    bucket, prefix = path.replace("s3://", "").split("/", 1)

    before, after = prefix.split("*/", 1)

    s3 = boto3.client("s3")
    response = s3.list_objects_v2(Bucket=bucket, Prefix=before, Delimiter="/")

    matches = [f"s3://{bucket}/{cp['Prefix']}{after}" for cp in response.get("CommonPrefixes", [])]

    if not matches:
        raise ValueError(f"No S3 paths matched: {path}")
    if len(matches) > 1:
        raise ValueError(f"Multiple matches: {matches}")

    return matches[0]

In [10]:
hist_ds = (
    catalog.get("pangeo-CESM2-WACCM-historical-icechunk")
    .to_xarray()
    .sel(ensemble_member="r2i1p1f1")
    .sel(lon=slice(25, 32), lat=slice(-33, -25))
).sel(time=slice("1978", "2015"))

In [11]:
var = "tas"

fname = get_fname(var=var, version="milestone01", scenario="historical", ens="r2i1p1f1")
fname = resolve_s3_glob(fname)
historical_downscaled_var = open_icechunk(path=fname)[var]

ValueError: No S3 paths matched: s3://carbonplan-scratch/srm/outputs/qa/milestone01/historical/CESM2-WACCM/tas/r2i1p1f1/lat-32.64to-26.58_lon25.05to31.39/*/historical.icechunk/